Only to ensure Onyxia GPUs to work.

In [6]:
# Onyxia GPU compatible torch version
import sys
!{sys.executable} -m pip install torch==2.11.0 torchaudio torchvision \
    --index-url https://download.pytorch.org/whl/cu126 \
    --force-reinstall

Looking in indexes: https://download.pytorch.org/whl/cu126
  Using cached torch-2.11.0%2Bcu126-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached torchaudio-2.11.0%2Bcu126-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached torchvision-0.26.0%2Bcu126-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-70.2.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_bindings-12.9.4-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 94.3 MB/s  0:00:

In [1]:
import pandas as pd
import os
import numpy as np
import re
import string
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


*If you want to analyse another sport, please change the path accordingly.*

In [ ]:
# ── 1. CHARGER LE CORPUS ──────────────────────────────────────────────────────

data_path = "PESSD/biath/"

df = pd.DataFrame()
for file in os.listdir(data_path):
    if file.endswith(".csv"):
        temp = pd.read_csv(os.path.join(data_path, file))
        df = pd.concat([df, temp])

In [34]:
# ── 2. NETTOYAGE ──────────────────────────────────────────────────────────────

df = df[(df["main_g"]=="female") | (df["main_g"]=="male")]
df["text"] = df["text"].apply(lambda x: x.split(". "))
df = df.explode("text")
df["text"] = df["text"].apply(lambda x: re.sub(r"\s*[A-Z]\w*\s*", " ", x).strip())
df["text"] = df["text"].apply(lambda x: x.translate(str.maketrans('', '', string.punctuation.replace("'", ""))))
df["ID"] = df["audio_file"].apply(lambda x: x.replace("_wav.wav", ""))
df = df.merge(pd.read_csv("metadata.csv"), on="ID")
df = df.reset_index(drop=True)


In [43]:
# ── 3. EXPLORER LE VOCABULAIRE RÉEL DU CORPUS ─────────────────────────────────

STOPWORDS = [x.strip() for x in open('stopwords.txt').readlines()]

words = " ".join(df["text"].tolist()).split()
words_filtered = [w for w in words if w not in STOPWORDS and len(w) > 2]
freq = Counter(words_filtered).most_common(50)

words_H = " ".join(df[df["g_ath"] == "H"]["text"].tolist()).split()
words_F = " ".join(df[df["g_ath"] == "F"]["text"].tolist()).split()

counter_H = Counter([w for w in words_H if w not in STOPWORDS and len(w) > 2])
counter_F = Counter([w for w in words_F if w not in STOPWORDS and len(w) > 2])

print("=== 50 mots les plus fréquents (hors stopwords) ===")
for word, count in freq:
    h = counter_H.get(word, 0)
    f = counter_F.get(word, 0)
    total = h + f
    pct_h = h / total * 100 if total > 0 else 0
    pct_f = f / total * 100 if total > 0 else 0
    print(f"  {word}: {count} (H: {pct_h:.0f}% | F: {pct_f:.0f}%)")

=== 50 mots les plus fréquents (hors stopwords) ===
  'est: 682 (H: 63% | F: 37%)
  secondes: 671 (H: 55% | F: 45%)
  tir: 456 (H: 59% | F: 41%)
  course: 306 (H: 57% | F: 43%)
  relais: 277 (H: 70% | F: 30%)
  faut: 247 (H: 63% | F: 37%)
  temps: 246 (H: 61% | F: 39%)
  médaille: 237 (H: 66% | F: 34%)
  vraiment: 234 (H: 67% | F: 33%)
  tête: 230 (H: 58% | F: 42%)
  petit: 227 (H: 59% | F: 41%)
  déjà: 215 (H: 57% | F: 43%)
  skis: 190 (H: 57% | F: 43%)
  également: 183 (H: 48% | F: 52%)
  vite: 182 (H: 50% | F: 50%)
  français: 176 (H: 85% | F: 15%)
  piste: 173 (H: 60% | F: 40%)
  peutêtre: 169 (H: 62% | F: 38%)
  monde: 167 (H: 60% | F: 40%)
  l'instant: 155 (H: 66% | F: 34%)
  voit: 147 (H: 69% | F: 31%)
  qu'on: 145 (H: 64% | F: 36%)
  chercher: 145 (H: 51% | F: 49%)
  olympique: 143 (H: 63% | F: 37%)
  tour: 134 (H: 63% | F: 37%)
  aller: 133 (H: 54% | F: 46%)
  coucher: 125 (H: 49% | F: 51%)
  train: 117 (H: 63% | F: 37%)
  sprint: 116 (H: 51% | F: 49%)
  n'est: 111 (H: 58% | F

In [ ]:
# ── 4. CALCULER LES EMBEDDINGS (GPU + cache) ──────────────────────────────────

embeddings_path = data_path+"embeddings.npy"

if os.path.exists(embeddings_path):
    print("\nChargement des embeddings depuis le cache...")
    embeddings = np.load(embeddings_path)
    print(f"Embeddings chargés : {embeddings.shape}")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nCalcul des embeddings sur : {device}")

    embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base", device=device)

    embeddings = embedding_model.encode(
        df["text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    np.save(embeddings_path, embeddings)
    print(f"Embeddings sauvegardés dans {embeddings_path}")

df["embedding"] = list(embeddings)
print(f"Corpus : {len(df)} phrases | embeddings : {embeddings.shape}")



Calcul des embeddings sur : cuda


Batches: 100%|██████████| 139/139 [00:10<00:00, 12.68it/s]


Embeddings sauvegardés dans data/embeddings.npy
Corpus : 8844 phrases | embeddings : (8844, 768)


In [40]:
# ── 5. DISTANCE COSINUS PAR TERME ─────────────────────────────────────────────

# Adapter cette liste après avoir regardé les 50 mots les plus fréquents
TERMES = ["tir", "course", "sprint", "olympique", "secondes", "coucher", "ski", "vraiment","médaille","fort","rapide"]

results = []

for terme in TERMES:
    mask_F = df["g_ath"].str.upper() == "F"
    mask_H = df["g_ath"].str.upper() == "H"
    mask_terme = df["text"].str.contains(terme, case=False, na=False)

    emb_F = np.array(df[mask_F & mask_terme]["embedding"].tolist())
    emb_H = np.array(df[mask_H & mask_terme]["embedding"].tolist())

    if len(emb_F) == 0 or len(emb_H) == 0:
        results.append({"terme": terme,  "similarite_cosinus": None})
        continue

    centroid_F = emb_F.mean(axis=0, keepdims=True)
    centroid_H = emb_H.mean(axis=0, keepdims=True)

    sim = cosine_similarity(centroid_F, centroid_H)[0][0]
    results.append({"terme": terme, "similarite_cosinus": round(sim, 4)})

# ── 6. RÉSULTATS ──────────────────────────────────────────────────────────────

results_df = pd.DataFrame(results).dropna(subset=["similarite_cosinus"]).sort_values("similarite_cosinus")
print("\n=== Similarité cosinus F vs H par terme ===")
print(results_df.to_string(index=False))


=== Similarité cosinus F vs H par terme ===
    terme  similarite_cosinus
     fort              0.9136
 vraiment              0.9216
   rapide              0.9361
olympique              0.9445
   sprint              0.9519
   course              0.9570
      ski              0.9649
  coucher              0.9711
      tir              0.9768
 médaille              0.9789
 secondes              0.9832
